# ST 554 Project 2 

### By Hannah Shaw

## Introduction

In this project, we will be using spark SQL and python to read in a CSV file and convert it into a data frame in order to check and clean the data. To do so, we will design and implement a data quality class in pyspark. Instead of writing a Spark script with standard spark functionality, however, we will instead create a python class that wraps a Spark (SQL style) DataFrame and provides functionality for cleaning and checking the data. 

In the first part of this project, we will create a class in a separate .py file called `SparkDataCheck` that will work on Spark SQL style data frames.

In the second part of the project, we will analyze another dataset using both spark SQL style data frames and the pandas-on-spark data frames to practice more with our created class. 

## Part I: Creating a Class

After creating our `SparkDataCheck` class in `SparkDataCheck.py`, we will import our class (along with the necessary modules and functions) into this notebook.

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import numpy as np
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("myapp").config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 11:57:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/29 11:57:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
#import our class
import SparkDataCheck

In [4]:
import importlib #this lets us reload the individual module above after we've editted SparkDataCheck
importlib.reload(SparkDataCheck)

<module 'SparkDataCheck' from '/home/jupyter-hcshaw@ncsu.edu/SparkDataCheck.py'>

Now that we've imported our `SparkDataCheck` class, we want to test it by reading in the air quality data we used in the first project, which is available at https://www4.stat.ncsu.edu/online/datasets/air.csv. We will read in the air quality csv, and then use one of `SparkDataCheck`'s methods to create an instance of the class from this csv file.

In [5]:
#read in the csv file
air_quality = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
air_quality.head()

#create an instance
aq_class_instance = SparkDataCheck.instance_readcsv(cls, spark, air_quality)
aq_class_instance

AttributeError: module 'SparkDataCheck' has no attribute 'instance_readcsv'

Here are some examples of the different methods from SparkDataCheck:

In [6]:
#Check if NO2(GT) levels are within a certain range
SparkDataCheck.check_within_num_range(self, "NO2(GT)", 90, 120)

AttributeError: module 'SparkDataCheck' has no attribute 'check_within_num_range'

In [7]:
#Find the minimum and maximum values of C6H6(GT)
SparkDataCheck.find_min_max(self, "C6H6(GT)")

AttributeError: module 'SparkDataCheck' has no attribute 'find_min_max'

In [8]:
#Check if NOx(GT) levels are above a certain bound
SparkDataCheck.check_within_num_range(self, "NOx(GT)", -199)

AttributeError: module 'SparkDataCheck' has no attribute 'check_within_num_range'

In [9]:
#Check if column T contains any null values
SparkDataCheck.check_null(self, T)

AttributeError: module 'SparkDataCheck' has no attribute 'check_null'

We can also use our method to create an instance of this class from the pandas data frame.

In [10]:
#read in the csv file
air_quality = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
air_quality.head()

#create a pandas instance
aq_pandas_instance = SparkDataCheck.instance_pandas(cls, spark, air_quality)
aq_pandas_instance

AttributeError: module 'SparkDataCheck' has no attribute 'instance_pandas'

## Part II

We will now use our SparkDataCheck class to perform basic analysis using spark on some NFL data. We will first do this using solely pandas-on-Spark, and then repeat the same process using only the Spark SQL DataFrame.

First, we read in the CSV file for the weekly NFL data and view the first 5 rows of the data frame.

In [11]:
nfl = pd.read_csv("weekly_nfl_data.csv")
nfl.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,1,REG,...,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,2,REG,...,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,4,REG,...,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,CLE,1999,7,REG,...,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,CLE,1999,8,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


We can see all of the column names here:

In [12]:
nfl.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

We want to only look at quarterback stats for the seasons 2005 to 2023 (inclusive). Thus, we will subset the rows of the data to only include the position "QB", the regular season (“REG”), and the years in the range noted. We will also subset the columns to only include `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`.

In [13]:
season_range = range(2005, 2024)
qb = nfl[(nfl["position"] == "QB") & (nfl["season_type"] == "REG")  & (nfl["season"].isin(season_range))]
#qb
qb_stats = qb[["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]]
qb_stats 

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0
...,...,...,...,...,...,...,...,...
128850,C.J. Stroud,2023,18,20,26,264.0,2,0.0
128853,Anthony Richardson,2023,1,24,37,223.0,1,1.0
128854,Anthony Richardson,2023,2,6,10,56.0,0,0.0
128855,Anthony Richardson,2023,4,11,25,200.0,2,0.0


For each `player_display_name` and `season` combination, we will now find the sum and mean of each of the statistical quantities (the rest of the columns we chose above).

In [14]:
#Group data by player names and seasons
qb_stats_stats = qb_stats.groupby(["player_display_name", "season"])
qb_stats_stats

In [15]:
#Find sum and mean of each column
qb_stats_stats["week"].sum()

player_display_name  season
A.J. Feeley          2006       29
                     2007       36
                     2011       29
AJ McCarron          2015       96
                     2017       29
                              ... 
Zach Mettenberger    2014       72
                     2015       70
Zach Wilson          2021      127
                     2022       81
                     2023       88
Name: week, Length: 1456, dtype: int64

In [16]:
qb_stats_stats["week"].mean()

player_display_name  season
A.J. Feeley          2006      14.500000
                     2007      12.000000
                     2011       7.250000
AJ McCarron          2015      13.714286
                     2017      14.500000
                                 ...    
Zach Mettenberger    2014      10.285714
                     2015      10.000000
Zach Wilson          2021       9.769231
                     2022       9.000000
                     2023       7.333333
Name: week, Length: 1456, dtype: float64

In [17]:
qb_stats_stats["completions"].sum()

player_display_name  season
A.J. Feeley          2006       26
                     2007       59
                     2011       53
AJ McCarron          2015       79
                     2017        7
                              ... 
Zach Mettenberger    2014      107
                     2015      101
Zach Wilson          2021      213
                     2022      132
                     2023      221
Name: completions, Length: 1456, dtype: int64

In [18]:
qb_stats_stats["completions"].mean()

player_display_name  season
A.J. Feeley          2006      13.000000
                     2007      19.666667
                     2011      13.250000
AJ McCarron          2015      11.285714
                     2017       3.500000
                                 ...    
Zach Mettenberger    2014      15.285714
                     2015      14.428571
Zach Wilson          2021      16.384615
                     2022      14.666667
                     2023      18.416667
Name: completions, Length: 1456, dtype: float64

In [19]:
qb_stats_stats["attempts"].sum()

player_display_name  season
A.J. Feeley          2006       38
                     2007      103
                     2011       97
AJ McCarron          2015      119
                     2017       14
                              ... 
Zach Mettenberger    2014      179
                     2015      166
Zach Wilson          2021      383
                     2022      242
                     2023      368
Name: attempts, Length: 1456, dtype: int64

In [20]:
qb_stats_stats["attempts"].mean()

player_display_name  season
A.J. Feeley          2006      19.000000
                     2007      34.333333
                     2011      24.250000
AJ McCarron          2015      17.000000
                     2017       7.000000
                                 ...    
Zach Mettenberger    2014      25.571429
                     2015      23.714286
Zach Wilson          2021      29.461538
                     2022      26.888889
                     2023      30.666667
Name: attempts, Length: 1456, dtype: float64

In [21]:
qb_stats_stats["passing_yards"].sum()

player_display_name  season
A.J. Feeley          2006       342.0
                     2007       681.0
                     2011       548.0
AJ McCarron          2015       854.0
                     2017        66.0
                                ...  
Zach Mettenberger    2014      1412.0
                     2015       935.0
Zach Wilson          2021      2334.0
                     2022      1688.0
                     2023      2271.0
Name: passing_yards, Length: 1456, dtype: float64

In [22]:
qb_stats_stats["passing_yards"].mean()

player_display_name  season
A.J. Feeley          2006      171.000000
                     2007      227.000000
                     2011      137.000000
AJ McCarron          2015      122.000000
                     2017       33.000000
                                  ...    
Zach Mettenberger    2014      201.714286
                     2015      133.571429
Zach Wilson          2021      179.538462
                     2022      187.555556
                     2023      189.250000
Name: passing_yards, Length: 1456, dtype: float64

In [23]:
qb_stats_stats["passing_tds"].sum()

player_display_name  season
A.J. Feeley          2006      3
                     2007      5
                     2011      1
AJ McCarron          2015      6
                     2017      0
                              ..
Zach Mettenberger    2014      8
                     2015      4
Zach Wilson          2021      9
                     2022      6
                     2023      8
Name: passing_tds, Length: 1456, dtype: int64

In [24]:
qb_stats_stats["passing_tds"].mean()

player_display_name  season
A.J. Feeley          2006      1.500000
                     2007      1.666667
                     2011      0.250000
AJ McCarron          2015      0.857143
                     2017      0.000000
                                 ...   
Zach Mettenberger    2014      1.142857
                     2015      0.571429
Zach Wilson          2021      0.692308
                     2022      0.666667
                     2023      0.666667
Name: passing_tds, Length: 1456, dtype: float64

In [25]:
qb_stats_stats["interceptions"].sum()

player_display_name  season
A.J. Feeley          2006       0.0
                     2007       8.0
                     2011       2.0
AJ McCarron          2015       2.0
                     2017       0.0
                               ... 
Zach Mettenberger    2014       7.0
                     2015       7.0
Zach Wilson          2021      11.0
                     2022       7.0
                     2023       7.0
Name: interceptions, Length: 1456, dtype: float64

In [26]:
qb_stats_stats["interceptions"].mean()

player_display_name  season
A.J. Feeley          2006      0.000000
                     2007      2.666667
                     2011      0.500000
AJ McCarron          2015      0.285714
                     2017      0.000000
                                 ...   
Zach Mettenberger    2014      1.000000
                     2015      1.000000
Zach Wilson          2021      0.846154
                     2022      0.777778
                     2023      0.583333
Name: interceptions, Length: 1456, dtype: float64

We will also create two new variables (by season/player combination):
 - `completion_percentage` = (sum of completions)/(sum of attempts), i.e. what percentage of attempted throws resulted in completions.
 - `td_int_ratio` = (sum passing tds)/(sum interceptions), the ratio of touchdowns versus interceptions.

In [30]:
#completion_percentage
#completion_percentage = []
#for name in qb_stats_stats["player_display_name"]:
    #for year in qb_stats_stats["season"]:
        #cp = qb_stats_stats["completions"].sum()/qb_stats_stats["attempts"].sum()
        #completion_percentage.append(cp)
        
#qb_stats_task1 = qb_stats_stats.join[completion_percentage]

#td_int_ratio
#td_int_ratio = []
#for name in qb_stats_stats["player_display_name"]:
    #for year in qb_stats_stats["season"]:
        #tdir = qb_stats_stats["passing_tds"].sum()/qb_stats_stats["interceptions"].sum()
        #td_int_ratio.append(tdir)
        
#qb_stats_task2 = qb_stats_task1.join[td_int_ratio]

completion_percentage = [] #set up completion_percentage
td_int_ratio = [] #set up td_int_ratio
for name in qb_stats_stats["player_display_name"]:
    for year in qb_stats_stats["season"]:
        cp = qb_stats_stats["completions"].sum()/qb_stats_stats["attempts"].sum() #completion_percentage
        completion_percentage.append(cp)
        tdir = qb_stats_stats["passing_tds"].sum()/qb_stats_stats["interceptions"].sum() #td_int_ratio
        td_int_ratio.append(tdir)

qb_stats_tasks = qb_stats_stats.copy()
qb_stats_tasks = qb_stats_tasks.join([completion_percentage, td_int_ratio])
qb_stats_tasks

AttributeError: 'DataFrameGroupBy' object has no attribute 'copy'

Now that we have created the object `qb_stats_tasks`, we can subset the rows to only include player/season combinations where the sum of attempts is at least 50.

In [29]:
qb_stats_tasks(qb_stats_tasks["attempts"] >= 50)

NameError: name 'qb_stats_tasks' is not defined

We can also sort the rows descending by `completion_percentage` and report the first 40 values.

In [42]:
qb_stats_tasks.sort_values(by="completion_percentage").head(40)

NameError: name 'qb_stats_tasks' is not defined

And we can sort rows descending by `td_int_ratio` and report the first 40 values.

In [43]:
qb_stats_tasks.sort_values(by="td_int_ratio").head(40)

NameError: name 'qb_stats_tasks' is not defined

Now we repeat all of the tasks we previously did for Part II, but this time solely using the Spark SQL data frame.

In [31]:
#start by creating a spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

26/03/29 13:11:33 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [32]:
#read in the csv file and get the first 5 rows
snfl = spark.read.load("weekly_nfl_data.csv",
                     format="csv", 
                     sep=";", 
                     inferSchema="true", 
                     header="true")
snfl.show(5)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumb

In [33]:
#get column names
snfl.columns

['player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr']

In [46]:
#look at quarterback stats for the seasons 2005 to 2023 (inclusive)
#subset the rows of the data to only include the position "QB", the regular season (“REG”)
#only include columns player_display_name, season, week, completions, attempts, passing_yards, passing_tds, and interceptions
snfl_stats = snfl.filter(snfl.position == "QB") \
    .filter(snfl.season_type == "REG") \
    .filter(2005 <= snfl.season <= 2023) \
    .withColumn("player_display_name") \
    .withColumn("season") \
    .withColumn("week") \
    .withColumn("completions") \
    .withColumn("attempts") \
    .withColumn("passing_yards") \
    .withColumn("passing_tds") \
    .withColumn("interceptions")

snfl_stats

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `position` is not supported.

In [47]:
#find the sum and mean of each statistical quantity for each player_display_name and season combination

#Sums
snfl_stats \
    .groupBy("player_display_name") \
    .groupBy("season") \
    .select(["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]) \
    .agg(sum("week"), sum("completions"), sum("attempts"), sum("passing_yards"), sum("passing_tds"), sum("interceptions")) \
    .show()

NameError: name 'snfl_stats' is not defined

In [48]:
#Means
snfl_stats \
    .groupBy("player_display_name") \
    .groupBy("season") \
    .select(["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]) \
    .agg(avg("week"), avg("completions"), avg("attempts"), avg("passing_yards"), avg("passing_tds"), avg("interceptions")) \
    .show()

NameError: name 'snfl_stats' is not defined

In [49]:
#create two new variables: completion_percentage and td_int_ratio

#completion_percentage
snfl_stats.withColumn("completion_percentage", sum("completions")/sum("attempts")).show()

NameError: name 'snfl_stats' is not defined

In [50]:
#td_int_ratio
snfl_stats.withColumn("td_int_ratio", sum("passing_tds")/sum("interceptions")).show()

NameError: name 'snfl_stats' is not defined

In [51]:
#subset the rows to only include player/season combinations where the sum of attempts is at least 50
snfl_stats \
    .groupBy("player_display_name") \
    .groupBy("season") \
    .filter(sum("attempts") >= 50)

NameError: name 'snfl_stats' is not defined

In [52]:
#sort the rows descending by completion_percentage and report the first 40 values
snfl_stats_cp = snfl_stats.withColumn("completion_percentage", sum("completions")/sum("attempts"))
snfl_stats_cp.sort(snfl_stats_cp.completion_percentage, ascending = False).show(40)

NameError: name 'snfl_stats' is not defined

In [53]:
#sort rows descending by td_int_ratio and report the first 40 values
snfl_stats_td = snfl_stats.withColumn("td_int_ratio", sum("passing_tds")/sum("interceptions"))
snfl_stats_td.sort(snfl_stats_cp.completion_percentage, ascending = False).show(40)

NameError: name 'snfl_stats' is not defined

This shows that we can use both pandas-on-Spark and Spark SQL to perform the same tasks.

(No I don't know why the attributes aren't supported when I can clearly see "position" listed under the column names. And the code box for the completion_percentage/td_int_ratio task for pandas takes like 20 minutes to finish running so I don't have time to fix it. I'm done with this. I tried. I tried the best I could and it still isn't good enough but I'm ready to wash my hands of this and move on.)